# Experiment 08 — History-dependent tipping window

**Question.** Does a past inflammatory episode lower the challenge needed to convert a site, and for how long does that vulnerability last after the episode has resolved?

**Setting.** We use the comparator backbone of Exp 06B with constant baseline plaque $u_b=0.20$. This baseline lies inside the bistable window $[0.063, 0.468]$, so healthy and diseased states coexist and a site's history decides which one it occupies. Every site starts healthy, on the low equilibrium at $u_b$.

**Protocol.** History → recovery at $u_b$ for $T_{\text{rec}}$ → acute challenge $u_{\text{ch}}$ lasting $D_{\text{ch}}$ → return to $u_b$ until the state has settled ($10/\varepsilon$). A site is *converted* if it ends on the high branch ($H>0.5$).

**Histories.**
* *naive*: no prior episode.
* *resolved gingivitis*: a sub-threshold episode, $u=0.45$ for $5/\varepsilon$. The site never leaves the low branch, because the system is monotone and approaches the low equilibrium at 0.45 from below.
* *treated disease*: the site has been on the high branch (memory $A$ at its high-state value, $\theta\approx0.30$). Therapy has returned $H$ and $M$ to their healthy values, but therapy does not act on $\theta$.

**Hosts.**
* Slow memory, $\varepsilon\in\{0.1,0.03,0.01\}$.
* Instant adaptation ($A=\alpha H$). This host has the same equilibria but no memory timescale, so for it the treated history is identical to the naive one.

**Sections**
* **A** example trajectories
* **B** critical challenge vs challenge duration for each history
* **C** the vulnerability window after treatment and after resolved gingivitis, measured in memory time
* **D** robustness to the baseline plaque level
* **E** checks and summary

In [ ]:
# =====================================================================
# EXPERIMENT 08 — folder setup
# =====================================================================
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/Biosemiotic_Perio")
except ImportError:                      # local execution
    BASE_DIR = Path.cwd() / "Biosemiotic_Perio"
EXP_DIR = BASE_DIR / "Experiment_08_Tipping_Window"
EXP_DIR.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("EXP_QUICK") == "1"     # set True for a fast smoke test with reduced grids
print(f"Experiment folder:\n{EXP_DIR}\nExists: {EXP_DIR.exists()}   QUICK={QUICK}")

In [ ]:
import sys, json, time, platform, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import brentq

versions_text = f"""
Experiment: 08 History-dependent tipping window
Created: {datetime.now(timezone.utc).isoformat()}

Python: {sys.version}
Platform: {platform.platform()}
NumPy: {np.__version__}
Pandas: {pd.__version__}
SciPy: {scipy.__version__}
Matplotlib: {matplotlib.__version__}
"""
with open(EXP_DIR / "software_versions.txt", "w") as f:
    f.write(versions_text)
print(versions_text)

# ---- figure style (same tokens as Experiment 07) ------------------------
INK, INK2, GRIDC, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
MEM_COL = {0.3: "#86b6ef", 0.1: "#5598e7", 0.03: "#2a78d6", 0.01: "#1c5cab", 0.003: "#0d366b"}
INSTANT_COL = "#eb6834"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.6,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "legend.frameon": False,
})
def save_fig(fig, name):
    fig.savefig(EXP_DIR / f"{name}.jpg", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =====================================================================
# CORE MODEL — state (H, M, A, D), A = ln(theta0/theta), D = structural memory
# =====================================================================
#   dH/dt = -H + F(M + u(t); theta0*exp(-A))
#   dM/dt =  r*(-M + G(H) + gamma*D)              pocket niche supports the community
#   dA/dt =  eps*(alpha*H - A)                     response-state memory ('adaptive')
#   dD/dt =  delta*max(H - hD, 0)*(1 - D)          irreversible, saturating tissue loss
# mode 'adaptive' (slow memory) | 'instant' (A = alpha*H: same equilibria, no memory
# timescale) | 'frozen' (A held fixed).  Fixed-step RK4, vectorised over elements.
COMPARATOR = dict(n=4.0, q=4.0, K=0.9, th0=0.8, r=1.0, a=1.0)   # Exp 06B backbone
DT = 0.1
U_UP, U_DOWN = 0.4682392215, 0.0630204768                       # static folds (D = 0)


def hill(x, h, n):
    x = np.maximum(x, 0.0)
    xn = x ** n
    return xn / (xn + h ** n)


def rhs(H, M, A, D, u, eps, p, mode, gamma, delta, hD):
    A_eff = p['a'] * H if mode == 'instant' else A
    dH = -H + hill(M + u, p['th0'] * np.exp(-A_eff), p['n'])
    dM = p['r'] * (-M + hill(H, p['K'], p['q']) + gamma * D)
    dA = eps * (p['a'] * H - A) if mode == 'adaptive' else 0.0 * A
    dD = delta * np.maximum(H - hD, 0.0) * (1.0 - D)
    return dH, dM, dA, dD


def run(y0, u_fn, n_steps, eps=0.0, mode='adaptive', p=COMPARATOR, gamma=0.0,
        delta=0.0, hD=0.8, events=None, recorder=None, dt=DT):
    """u_fn(i, c): forcing for step i at stage offset c in {0, .5, 1};
    events(i, H, M, A, D) -> state, applied at the start of step i;
    recorder(i, H, M, A, D), called after each step."""
    H, M, A, D = [np.array(v, float, copy=True) for v in np.broadcast_arrays(*y0)]
    eps, gamma, delta = [np.broadcast_to(np.asarray(v, float), H.shape) for v in (eps, gamma, delta)]
    if isinstance(p.get('r'), np.ndarray):
        p = dict(p, r=np.broadcast_to(p['r'], H.shape))
    f = lambda h, m, a, d, u: rhs(h, m, a, d, u, eps, p, mode, gamma, delta, hD)
    for i in range(n_steps):
        if events is not None:
            H, M, A, D = events(i, H, M, A, D)
        u0, uh, u1 = u_fn(i, 0.0), u_fn(i, 0.5), u_fn(i, 1.0)
        k1 = f(H, M, A, D, u0)
        k2 = f(H + .5*dt*k1[0], M + .5*dt*k1[1], A + .5*dt*k1[2], D + .5*dt*k1[3], uh)
        k3 = f(H + .5*dt*k2[0], M + .5*dt*k2[1], A + .5*dt*k2[2], D + .5*dt*k2[3], uh)
        k4 = f(H + dt*k3[0], M + dt*k3[1], A + dt*k3[2], D + dt*k3[3], u1)
        H = H + dt/6 * (k1[0] + 2*k2[0] + 2*k3[0] + k4[0])
        M = M + dt/6 * (k1[1] + 2*k2[1] + 2*k3[1] + k4[1])
        A = A + dt/6 * (k1[2] + 2*k2[2] + 2*k3[2] + k4[2])
        D = D + dt/6 * (k1[3] + 2*k2[3] + 2*k3[3] + k4[3])
        if recorder is not None:
            recorder(i, H, M, A, D)
    if mode == 'instant':
        A = p['a'] * H
    return H, M, A, D


def piecewise_u(boundaries, values):
    """Piecewise-constant forcing: segment k+1 starts at step boundaries[k] (per element)."""
    B = np.stack([np.asarray(b, np.int64) for b in boundaries])
    V = np.stack([np.asarray(v, float) for v in values])
    def u_fn(i, c):
        seg = (i >= B).sum(axis=0)
        return np.take_along_axis(V, seg[None, :], 0)[0]
    return u_fn


def u_branch(H, p=COMPARATOR, gamma=0.0, D=0.0):
    """Equilibrium forcing as a function of H* (M1 with structural memory D frozen)."""
    th = p['th0'] * np.exp(-p['a'] * H)
    return th * (H / (1 - H)) ** (1 / p['n']) - hill(H, p['K'], p['q']) - gamma * D


def low_equilibrium(u, p=COMPARATOR, gamma=0.0, D=0.0):
    Hs = np.linspace(1e-9, 0.6, 20001)
    f = u_branch(Hs, p, gamma, D) - u
    i = np.where(np.sign(f[:-1]) != np.sign(f[1:]))[0][0]
    H = brentq(lambda h: u_branch(h, p, gamma, D) - u, Hs[i], Hs[i + 1], xtol=1e-15)
    return H, hill(H, p['K'], p['q']) + gamma * D, p['a'] * H


def high_equilibrium(u, p=COMPARATOR, gamma=0.0, D=0.0):
    Hs = 1 - np.logspace(-12, np.log10(0.5), 20001)
    f = u_branch(Hs, p, gamma, D) - u
    i = np.where(np.sign(f[:-1]) != np.sign(f[1:]))[0][0]
    H = brentq(lambda h: u_branch(h, p, gamma, D) - u, Hs[i + 1], Hs[i], xtol=1e-15)
    return H, hill(H, p['K'], p['q']) + gamma * D, p['a'] * H

In [ ]:
# =====================================================================
# PARAMETERS AND PROTOCOL
# =====================================================================
UB = 0.20            # baseline plaque (inside the bistable window)
U_GING = 0.45        # resolved-gingivitis episode (below the static fold 0.468)
U_CH_MAX = 4.0       # largest challenge tested
HOSTS = [('instant', 0.0), ('adaptive', 0.1), ('adaptive', 0.03), ('adaptive', 0.01)]
D_CH_GRID = np.array([5, 10, 20, 50, 100, 200, 500], float)
REC_X = np.array([0, 0.1, 0.25, 0.5, 1, 1.5, 2, 3, 4, 6, 8], float)     # eps * T_rec
REC_INSTANT = np.array([0, 0.5, 1, 2, 5, 10, 20, 50], float)            # absolute, instant host
UB_ROBUST = [0.10, 0.20, 0.30]
ROUNDS, NPTS = 3, 17
if QUICK:
    HOSTS = HOSTS[:2]; D_CH_GRID = np.array([10, 50], float)
    REC_X = np.array([0, 1, 4], float); REC_INSTANT = np.array([0, 5], float)
    UB_ROBUST = [0.20]; ROUNDS, NPTS = 2, 9

host_name = lambda mode, eps: 'instant' if mode == 'instant' else f'eps={eps}'
EQ = {}
def equilibria(ub):
    if ub not in EQ:
        EQ[ub] = dict(low=low_equilibrium(ub), high=high_equilibrium(ub))
    return EQ[ub]

with open(EXP_DIR / "parameters.json", "w") as f:
    json.dump(dict(backbone=COMPARATOR, u_b=UB, u_gingivitis=U_GING, u_ch_max=U_CH_MAX,
                   hosts=[host_name(*h) for h in HOSTS], D_ch_grid=D_CH_GRID.tolist(),
                   eps_T_rec=REC_X.tolist(), T_rec_instant=REC_INSTANT.tolist(), u_b_robust=UB_ROBUST,
                   dt=DT, conversion='H > 0.5 after settling for 10/eps (instant: 200)',
                   low_eq_u_b=equilibria(UB)['low'], high_eq_u_b=equilibria(UB)['high']), f, indent=4)
print("low equilibrium at u_b :", np.round(equilibria(UB)['low'], 5))
print("high equilibrium at u_b:", np.round(equilibria(UB)['high'], 5),
      " theta =", round(COMPARATOR['th0'] * np.exp(-equilibria(UB)['high'][2]), 4))


def challenge_outcome(uch, Dch, Trec, eps, mode, history, ub=UB, recorder=None):
    """Final H after history -> recovery -> challenge -> settling (arrays broadcast)."""
    uch, Dch, Trec = [np.ravel(v).astype(float) for v in np.broadcast_arrays(uch, Dch, Trec)]
    n = uch.size
    Hl, Ml, Al = equilibria(ub)['low']
    A_hist = Al
    Dcond, ucond = 0.0, ub
    if history == 'gingivitis':
        Dcond, ucond = (5 / eps if mode == 'adaptive' else 50.0), U_GING
    elif history == 'treated' and mode == 'adaptive':
        A_hist = equilibria(ub)['high'][2]          # memory of the diseased state retained
    settle = 10 / eps if mode == 'adaptive' else 200.0
    b1 = np.full(n, int(round(Dcond / DT)))
    b2 = b1 + np.rint(Trec / DT).astype(np.int64)
    b3 = b2 + np.rint(Dch / DT).astype(np.int64)
    u_fn = piecewise_u([b1, b2, b3], [np.full(n, ucond), np.full(n, ub), uch, np.full(n, ub)])
    H, M, A, D = run((np.full(n, Hl), Ml, A_hist, 0.0), u_fn, int(b3.max() + round(settle / DT)),
                     eps=eps, mode=mode, recorder=recorder)
    return H


def critical_challenge(Dch, Trec, eps, mode, history, ub=UB):
    """Smallest challenge amplitude that converts the site (vectorised grid refinement).
    NaN: no conversion up to U_CH_MAX.  Equal to ub: converts without any challenge."""
    Dch, Trec = [np.ravel(v).astype(float) for v in np.broadcast_arrays(Dch, Trec)]
    m = Dch.size; ar = np.arange(m)
    lo, hi = np.full(m, ub), np.full(m, U_CH_MAX)
    conv_hi = challenge_outcome(hi, Dch, Trec, eps, mode, history, ub) > 0.5
    conv_lo = challenge_outcome(lo, Dch, Trec, eps, mode, history, ub) > 0.5
    for _ in range(ROUNDS):
        g = np.linspace(lo, hi, NPTS)
        conv = challenge_outcome(g, np.broadcast_to(Dch, g.shape), np.broadcast_to(Trec, g.shape),
                                 eps, mode, history, ub).reshape(g.shape) > 0.5
        idx = np.argmax(conv, axis=0)
        lo = np.where(idx > 0, g[np.maximum(idx - 1, 0), ar], lo)
        hi = g[idx, ar]
    out = np.where(conv_hi, 0.5 * (lo + hi), np.nan)
    out[conv_lo] = ub
    return out

## A. Example trajectories

The same acute challenge ($u_{\text{ch}}=0.45$ for $2/\varepsilon$, with $\varepsilon=0.03$) is applied to four sites:
* a naive site;
* a treated site immediately after therapy;
* a treated site after recovering for $4/\varepsilon$;
* the instant host after treatment.

In [ ]:
EPS_A = 0.03 if not QUICK else 0.1
DCH_A, UCH_A = round(2 / EPS_A, 1), 0.45
scen = [('naive', 'adaptive', EPS_A, 0.0), ('treated', 'adaptive', EPS_A, 0.0),
        ('treated', 'adaptive', EPS_A, 4 / EPS_A), ('treated', 'instant', 0.0, 0.0)]
rows = []
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
styles = ['-', '-', '--', '-']
cols = [INK2, MEM_COL[0.01], MEM_COL[0.1], INSTANT_COL]
for (hist, mode, eps, Tr), ls, col in zip(scen, styles, cols):
    tr = []
    rec = lambda i, H, M, A, D: tr.append((i * DT, H[0], A[0])) if i % 10 == 0 else None
    Hf = challenge_outcome(UCH_A, DCH_A, Tr, eps, mode, hist, recorder=rec)
    tr = np.array(tr)
    t0 = (5 / eps if (hist == 'gingivitis') else 0.0) + Tr              # challenge onset
    th = COMPARATOR['th0'] * np.exp(-(tr[:, 2] if mode == 'adaptive' else COMPARATOR['a'] * tr[:, 1]))
    lab = f"{hist}, {host_name(mode, eps)}, recovery {Tr:g}" + (" -> converts" if Hf[0] > 0.5 else " -> resists")
    ax[0].plot(tr[:, 0] - t0, tr[:, 1], ls, color=col, label=lab)
    ax[1].plot(tr[:, 0] - t0, th, ls, color=col)
    for t_, h_, th_ in zip(tr[:, 0] - t0, tr[:, 1], th):
        rows.append(dict(scenario=lab, time_from_challenge=t_, H=h_, theta=th_))
for a_ in ax:
    a_.axvspan(0, DCH_A, color=GRIDC, alpha=0.6, lw=0)
    a_.set_xlim(-20, DCH_A + 8 / EPS_A)
ax[0].set(xlabel="time from challenge onset", ylabel="host response H",
          title=f"Same challenge ({UCH_A} for {DCH_A:g}), different histories")
ax[1].set(xlabel="time from challenge onset", ylabel=r"threshold $\theta$", title="What the history left behind")
ax[0].legend(fontsize=6.5, loc="center right")
pd.DataFrame(rows).to_csv(EXP_DIR / "fig08_01_example_trajectories_data.csv", index=False)
save_fig(fig, "fig08_01_example_trajectories")

## B. Critical challenge vs challenge duration

For each host and history, we find the smallest challenge amplitude that converts the site at $T_{\text{rec}}=0$. NaN means that no challenge up to amplitude 4 converts the site.

In [ ]:
HIST = ['naive', 'gingivitis', 'treated']
HIST_COL = {'naive': INK2, 'gingivitis': "#1baf7a", 'treated': "#6250d6"}
HIST_LAB = {'naive': 'naive', 'gingivitis': 'resolved gingivitis', 'treated': 'treated disease'}
rows = []
for mode, eps in HOSTS:
    t0 = time.time()
    for hist in HIST:
        uc = critical_challenge(D_CH_GRID, 0.0, eps, mode, hist)
        for d_, u_ in zip(D_CH_GRID, uc):
            rows.append(dict(host=host_name(mode, eps), history=hist, D_ch=d_, critical_u_ch=u_))
    print(f"{host_name(mode, eps):9s} done ({time.time() - t0:.0f}s)")
dfB = pd.DataFrame(rows); dfB.to_csv(EXP_DIR / "fig08_02_critical_challenge_data.csv", index=False)
display(dfB.pivot_table(index=['host', 'history'], columns='D_ch', values='critical_u_ch', dropna=False).round(3))

fig, axs = plt.subplots(1, len(HOSTS), figsize=(2.6 * len(HOSTS) + 1, 3.3), constrained_layout=True, sharey=True)
axs = np.atleast_1d(axs)
for a_, (mode, eps) in zip(axs, HOSTS):
    d = dfB[dfB.host == host_name(mode, eps)]
    for hist in HIST:
        dd = d[d.history == hist]
        y = dd.critical_u_ch.values
        a_.plot(dd.D_ch, np.where(np.isnan(y), np.nan, y), 'o-' if hist != 'naive' else 'o--',
                ms=3.5, color=HIST_COL[hist], label=HIST_LAB[hist])
        nn = np.isnan(y)
        a_.plot(dd.D_ch[nn], np.full(nn.sum(), 4.4), 'v', ms=5, color=HIST_COL[hist], mfc='none')
    a_.axhline(U_UP, color=INK2, lw=0.9, ls=':'); a_.axhline(UB, color=INK2, lw=0.9)
    a_.set(xscale='log', yscale='log', ylim=(0.17, 4.8), xlabel="challenge duration",
           title="instant adaptation" if mode == 'instant' else rf"slow memory, $\varepsilon$ = {eps}")
axs[0].set_ylabel("critical challenge amplitude")
axs[0].legend(fontsize=7, loc="upper right")
axs[-1].text(0.98, 0.02, "dotted: static fold   solid: baseline\nopen triangles: no conversion up to 4",
             transform=axs[-1].transAxes, ha='right', va='bottom', fontsize=6.5, color=INK2)
save_fig(fig, "fig08_02_critical_challenge")

## C. The vulnerability window

After treatment (and after resolved gingivitis), the critical challenge is measured as a function of the recovery time $T_{\text{rec}}$, with challenge duration $D_{\text{ch}}=2/\varepsilon$. For the instant host, $D_{\text{ch}}=20$ and $T_{\text{rec}}$ is in absolute time.

Normalised vulnerability:

$$V(T_{\text{rec}})=\frac{u^*_{\text{naive}}-u^*(T_{\text{rec}})}{u^*_{\text{naive}}-u^*(0)}$$

This is compared with the decay of the memory variable itself, $A(T_{\text{rec}})/A(0)$, and with passive decay $e^{-\varepsilon T_{\text{rec}}}$, whose half-life is $\ln 2$ in memory-time units.

In [ ]:
rows, mem_rows = [], []
for mode, eps in HOSTS:
    t0 = time.time()
    if mode == 'instant':
        Dch, Trec = 20.0, REC_INSTANT
    else:
        Dch, Trec = round(2 / eps, 1), REC_X / eps
    u_naive = critical_challenge(np.array([Dch]), 0.0, eps, mode, 'naive')[0]
    for hist in ['treated', 'gingivitis']:
        uc = critical_challenge(np.full(len(Trec), Dch), Trec, eps, mode, hist)
        for tr_, u_ in zip(Trec, uc):
            rows.append(dict(host=host_name(mode, eps), history=hist, D_ch=Dch, T_rec=tr_,
                             eps_T_rec=tr_ * eps if mode == 'adaptive' else np.nan,
                             critical_u_ch=u_, critical_u_naive=u_naive))
    if mode == 'adaptive':                                  # memory left at challenge onset (treated)
        Hl, Ml, Al = equilibria(UB)['low']; Ah = equilibria(UB)['high'][2]
        snap = {}
        steps = np.rint(Trec / DT).astype(int); step_set = set(steps.tolist())
        rec = lambda i, H, M, A, D: snap.__setitem__(i + 1, A[0]) if (i + 1) in step_set else None
        run((np.array([Hl]), Ml, Ah, 0.0), lambda i, c: np.array([UB]), int(steps.max()), eps=eps, recorder=rec)
        snap[0] = Ah
        for tr_, s_ in zip(Trec, steps):
            mem_rows.append(dict(host=host_name(mode, eps), eps_T_rec=tr_ * eps,
                                 A_rel=(snap[s_] - Al) / (Ah - Al)))
    print(f"{host_name(mode, eps):9s} done ({time.time() - t0:.0f}s)")
dfC = pd.DataFrame(rows)
dfC['V'] = np.nan
for (h, hist), g in dfC.groupby(['host', 'history']):
    u0 = g.sort_values('T_rec').critical_u_ch.iloc[0]
    dfC.loc[g.index, 'V'] = (g.critical_u_naive - g.critical_u_ch) / (g.critical_u_naive - u0)
dfC.to_csv(EXP_DIR / "fig08_03_vulnerability_window_data.csv", index=False)
dfM = pd.DataFrame(mem_rows); dfM.to_csv(EXP_DIR / "memory_decay_after_treatment.csv", index=False)

def half_point(x, y):
    """first x where y falls to 0.5 (linear interpolation); NaN if never"""
    x, y = np.asarray(x), np.asarray(y)
    i = np.where(y <= 0.5)[0]
    if i.size == 0 or i[0] == 0:
        return np.nan
    j = i[0]
    return x[j - 1] + (0.5 - y[j - 1]) * (x[j] - x[j - 1]) / (y[j] - y[j - 1])

hl = []
for (h, hist), g in dfC[dfC.host != 'instant'].groupby(['host', 'history']):
    g = g.sort_values('eps_T_rec')
    hl.append(dict(host=h, history=hist, vulnerability_half_life_memory_units=half_point(g.eps_T_rec, g.V)))
for h, g in dfM.groupby('host'):
    hl.append(dict(host=h, history='treated (memory variable A)',
                   vulnerability_half_life_memory_units=half_point(g.eps_T_rec, g.A_rel)))
hl.append(dict(host='passive decay', history='exp(-eps t)', vulnerability_half_life_memory_units=np.log(2)))
dfH = pd.DataFrame(hl); dfH.to_csv(EXP_DIR / "vulnerability_half_lives.csv", index=False)
display(dfH.round(3))

# window length for fixed challenges: the eps*T_rec at which u*(T_rec) rises past c * u*_naive
wl = []
for (h, hist), g in dfC[dfC.host != 'instant'].groupby(['host', 'history']):
    g = g.sort_values('eps_T_rec')
    for cf in [0.5, 0.6, 0.7, 0.8, 0.9]:
        c = cf * g.critical_u_naive.iloc[0]; y = g.critical_u_ch.values
        k = np.where(y >= c)[0]
        W = np.nan if k.size == 0 else (0.0 if k[0] == 0 else
            g.eps_T_rec.values[k[0]-1] + (c - y[k[0]-1]) * (g.eps_T_rec.values[k[0]] - g.eps_T_rec.values[k[0]-1]) / (y[k[0]] - y[k[0]-1]))
        wl.append(dict(host=h, history=hist, challenge_fraction_of_naive_threshold=cf, window_memory_units=W))
dfW = pd.DataFrame(wl); dfW.to_csv(EXP_DIR / "window_lengths.csv", index=False)
display(dfW.pivot_table(index=['history', 'challenge_fraction_of_naive_threshold'], columns='host',
                        values='window_memory_units').round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.5), constrained_layout=True)
for mode, eps in HOSTS:
    if mode == 'instant':
        continue
    h = host_name(mode, eps)
    for hist, ls in [('treated', '-'), ('gingivitis', '--')]:
        g = dfC[(dfC.host == h) & (dfC.history == hist)].sort_values('eps_T_rec')
        ax[0].plot(g.eps_T_rec, g.critical_u_ch / g.critical_u_naive, ls + 'o', ms=3, color=MEM_COL[eps],
                   label=f"{HIST_LAB[hist]}, " + rf"$\varepsilon$={eps}")
        ax[1].plot(g.eps_T_rec, g.V, ls + 'o', ms=3, color=MEM_COL[eps])
    gm = dfM[dfM.host == h].sort_values('eps_T_rec')
    ax[1].plot(gm.eps_T_rec, gm.A_rel, ':', color=MEM_COL[eps], lw=1.2)
xx = np.linspace(0, REC_X.max(), 200)
ax[1].plot(xx, np.exp(-xx), color=INK, lw=1, label=r"passive decay $e^{-\varepsilon T}$")
gi = dfC[(dfC.host == 'instant') & (dfC.history == 'treated')]
ax[0].axhline(1.0, color=INK2, lw=0.9, ls=':')
ax[0].set(xlabel=r"recovery time  $\varepsilon\,T_{rec}$  (memory-time units)",
          ylabel="critical challenge / naive", title="Vulnerability after an episode")
ax[1].set(xlabel=r"recovery time  $\varepsilon\,T_{rec}$", ylabel="normalised vulnerability V",
          title="Window outlasts passive memory decay")
ax[0].legend(fontsize=6.5, loc="lower right")
ax[1].text(0.97, 0.75, "solid/dashed: V (treated / gingivitis)\ndotted: memory variable A\n"
           f"instant host, treated: V = 0 at all T_rec\n(u* = {gi.critical_u_ch.min():.3f}-{gi.critical_u_ch.max():.3f}, naive {gi.critical_u_naive.iloc[0]:.3f})",
           transform=ax[1].transAxes, ha='right', fontsize=6.5, color=INK2)
ax[1].legend(fontsize=7, loc="upper right")
save_fig(fig, "fig08_03_vulnerability_window")

## D. Robustness to the baseline plaque level

The treated history is rerun with $\varepsilon=0.03$ at baseline levels $u_b\in\{0.10, 0.20, 0.30\}$. Equilibria and memory are recomputed for each baseline.

In [ ]:
EPS_D = 0.03 if not QUICK else 0.1
DCH_D = round(2 / EPS_D, 1)
rows = []
for ub in UB_ROBUST:
    t0 = time.time()
    un = critical_challenge(np.array([DCH_D]), 0.0, EPS_D, 'adaptive', 'naive', ub=ub)[0]
    uc = critical_challenge(np.full(len(REC_X), DCH_D), REC_X / EPS_D, EPS_D, 'adaptive', 'treated', ub=ub)
    for x_, u_ in zip(REC_X, uc):
        rows.append(dict(u_b=ub, eps_T_rec=x_, critical_u_ch=u_, critical_u_naive=un))
    print(f"u_b={ub} done ({time.time() - t0:.0f}s)")
dfD = pd.DataFrame(rows)
dfD['V'] = np.nan
for ub, g in dfD.groupby('u_b'):
    dfD.loc[g.index, 'V'] = (g.critical_u_naive - g.critical_u_ch) / (g.critical_u_naive - g.critical_u_ch.iloc[0])
dfD.to_csv(EXP_DIR / "fig08_04_baseline_robustness_data.csv", index=False)
fig, ax = plt.subplots(figsize=(4.6, 3.3), constrained_layout=True)
for ub, ls in zip(UB_ROBUST, ['--', '-', ':']):
    g = dfD[dfD.u_b == ub]
    ax.plot(g.eps_T_rec, g.V, ls + 'o', ms=3, color=MEM_COL[EPS_D], label=f"baseline $u_b$ = {ub}")
ax.plot(xx, np.exp(-xx), color=INK, lw=1, label="passive decay")
ax.set(xlabel=r"recovery time  $\varepsilon\,T_{rec}$", ylabel="normalised vulnerability V",
       title=rf"Treated site, $\varepsilon$ = {EPS_D}")
ax.legend(fontsize=7)
save_fig(fig, "fig08_04_baseline_robustness")
display(dfD.round(3))

## E. Consistency checks and summary

1. Conversion must be monotone in challenge amplitude, which is what the threshold search assumes. This is checked on a coarse grid for every host and history.
2. The critical challenge should not decrease with recovery time.
3. A site must never convert without a challenge.

In [ ]:
chk = []
for mode, eps in HOSTS:
    Dch = 20.0 if mode == 'instant' else round(2 / eps, 1)
    for hist in HIST:
        g = np.linspace(UB, U_CH_MAX, 25)
        conv = challenge_outcome(g, Dch, 0.0, eps, mode, hist) > 0.5
        mono = bool(np.all(np.diff(conv.astype(int)) >= 0))
        none = bool(challenge_outcome(np.array([UB]), Dch, 0.0, eps, mode, hist)[0] <= 0.5)
        chk.append(dict(host=host_name(mode, eps), history=hist, monotone_in_amplitude=mono,
                        no_conversion_without_challenge=none))
for (h, hist), g in dfC.groupby(['host', 'history']):
    y = g.sort_values('T_rec').critical_u_ch.values; y = y[~np.isnan(y)]
    chk.append(dict(host=h, history=hist, critical_nondecreasing_in_T_rec=bool(np.all(np.diff(y) >= -2e-3))))
dfE = pd.DataFrame(chk); dfE.to_csv(EXP_DIR / "consistency_checks.csv", index=False)
display(dfE)

summary = []
for (h, hist), g in dfC.groupby(['host', 'history']):
    g = g.sort_values('T_rec')
    summary.append(dict(host=h, history=hist, D_ch=g.D_ch.iloc[0], naive_threshold=g.critical_u_naive.iloc[0],
                        threshold_right_after=g.critical_u_ch.iloc[0],
                        relative_drop=1 - g.critical_u_ch.iloc[0] / g.critical_u_naive.iloc[0]))
dfS = pd.DataFrame(summary).merge(dfH, on=['host', 'history'], how='left')
dfS.to_csv(EXP_DIR / "summary.csv", index=False)
display(dfS.round(3))

In [ ]:
readme = """Experiment 08 - History-dependent tipping window
================================================
Comparator backbone (Exp 06B), baseline plaque u_b = 0.20 (bistable window).
Protocol: history -> recovery at u_b (T_rec) -> challenge u_ch for D_ch -> settle 10/eps.
Histories: naive | resolved gingivitis (u = 0.45 for 5/eps) | treated disease (H, M healthy,
memory A retained from the high state). Hosts: slow memory eps = 0.1, 0.03, 0.01; instant
adaptation (same equilibria). Converted: H > 0.5 after settling.

Files
- parameters.json, software_versions.txt
- fig08_01_example_trajectories(.jpg/_data.csv)   same challenge, four histories/hosts
- fig08_02_critical_challenge                      critical amplitude vs challenge duration
- fig08_03_vulnerability_window                    critical challenge and V vs eps*T_rec
- memory_decay_after_treatment.csv                 A(T_rec) relative to its post-treatment value
- vulnerability_half_lives.csv, window_lengths.csv
- fig08_04_baseline_robustness                     u_b = 0.10 / 0.20 / 0.30, eps = 0.03
- consistency_checks.csv, summary.csv
"""
with open(EXP_DIR / "README.txt", "w") as f:
    f.write(readme)
archive = shutil.make_archive(str(BASE_DIR / "Experiment_08_Tipping_Window_EXPORT"), 'zip', EXP_DIR)
print("Saved:", archive)